## Add liver cell annotations to GSE256398

In [ ]:
from pathlib import Path

import pandas as pd
import scanpy as sc
import anndata as ad
from scipy import sparse
from scipy.stats import median_abs_deviation
import numpy as np
import celltypist
from celltypist import models

In [ ]:
base_dir = Path(r"path/to/your/directory")  # Change to your working directory
output_h5ad = base_dir / "GSE256398_selected_raw.h5ad"

In [ ]:
# =========================================================
# Read in adata
# =========================================================
print(f"\n[INFO] Reading back saved file: {output_h5ad}")
adata = sc.read_h5ad(output_h5ad)
print(adata)
print("\n[INFO] Patient cell counts from saved file:")
print(adata.obs["patient_id"].value_counts())

In [4]:
model = models.Model.load(model = 'Healthy_Human_Liver.pkl')

In [5]:
# Save the counts layer to a new layer called "raw_counts"
adata.layers["raw_counts"] = adata.X.copy()

In [6]:
# Normalize to 10,000 counts per cell
sc.pp.normalize_total(adata, target_sum=1e4)

# Natural log1p transform
sc.pp.log1p(adata)

In [ ]:
predictions = celltypist.annotate(adata, model = 'Healthy_Human_Liver.pkl', majority_voting = True)
adata = predictions.to_adata(prefix="typist_liver_", insert_conf_by="majority_voting")
print(adata.obs["typist_liver_majority_voting"].head())
print(adata.obs["typist_liver_conf_score"].head())

In [8]:
score_col = "typist_liver_conf_score"

# Print head of this column
print(adata.obs[score_col].head())
# Print number of cells in adata
print(adata.n_obs)

# Filter adata for score_col > 0.5
adata = adata[adata.obs[score_col] > 0.5]
print(f"After filtering for {score_col} > 0.5, number of cells in adata: {adata.n_obs}")


GSM8097080_TGAGAGGCAGCTTCGG-1    1.0
GSM8097080_AGATCTGCACGTCAGC-1    1.0
GSM8097080_GTAACTGAGTGACATA-1    1.0
GSM8097080_GTGAAGGGTCCGTTAA-1    1.0
GSM8097080_AAGGCAGCAGATGAGC-1    1.0
Name: typist_liver_conf_score, dtype: float64
116197
After filtering for typist_liver_conf_score > 0.5, number of cells in adata: 104895


In [9]:
# See the counts of typist_liver_majority_voting
print(adata.obs["typist_liver_majority_voting"].value_counts())

typist_liver_majority_voting
Hepatocytes                65127
Fibroblasts                 9852
Endothelial cells           9140
T cells                     6771
Cholangiocytes              6394
Macrophages                 5496
Neutrophils                  627
Plasma cells                 504
Resident NK                  388
B cells                      288
Circulating NK/NKT           155
Basophils                     54
cDC1s                         52
pDCs                          28
Mono+mono derived cells       19
Name: count, dtype: int64


In [12]:
cell_types_to_adata = [
    "Cholangiocytes",
    "Hepatocytes",
    "Fibroblasts",
    "Endothelial cells",
    "Macrophages",
    "T cells",
    "Resident NK"
]

liver_dir = base_dir / "liver"
liver_dir.mkdir(exist_ok=True, parents=True)

existing_labels = set(adata.obs["typist_liver_majority_voting"].astype(str).unique())
celltype_adatas = {}

for cell_type in cell_types_to_adata:
    if cell_type not in existing_labels:
        continue

    subset = adata[adata.obs["typist_liver_majority_voting"] == cell_type].copy()
    celltype_adatas[cell_type] = subset

    safe_name = (
        cell_type.replace("/", "_")
        .replace(" ", "_")
    )
    subset.write_h5ad(liver_dir / f"{safe_name}.h5ad")

In [ ]:
# =========================================================
# Save AnnData object to disk
# =========================================================
output_h5ad = base_dir / "GSE256398_selected_annotated.h5ad"
print(f"\n[INFO] Saving to: {output_h5ad}")
output_h5ad.parent.mkdir(parents=True, exist_ok=True)
adata.write_h5ad(output_h5ad, compression="gzip")
print("[INFO] Done.")